# **Project: Producct Analystics E-Commerce**
### Tool   : Goggle Colab
### Dataset : UCI Online Retail
### Goal : **Product Analytics : Customer Retention & Revenue Drivers in E-Commerce**

####Product : Online Retailer and digital marketplace
####Users   : Online shoppers or e-consumers
####Problem : Understand user retention, repeat purchase and average revenue per customer
####Core user action : Completed Purchase(Order Placed)
####NSM : Average revenue per active customer

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
df = pd.read_excel("/kaggle/input/uci-online-retail-dataset/Online Retail.xlsx")

In [ ]:
df.head()

In [ ]:
df.shape

Initial data loaded **Successfully**.
###Next Step : Data undestanding and cleaning.

### Data cleaning & preparation

*   Data cleaning is important to insure accuracy, reliability and consistency of insights.

*   There can be Null values like missing customer ID/Invoice no/stock code, order cancellation , negative quantity or negative pricing.

*   That can affect Average order value, Average order quantity like metrics.




In [ ]:
df.CustomerID.isnull().sum()

In [ ]:
df = df.dropna(subset=['CustomerID']) #You can not calculate retention or revenue per user without user id

In [ ]:
df.CustomerID.isnull().sum()

In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df[df['InvoiceNo'].str.startswith('C')].shape

In [ ]:
df.drop(df[df['InvoiceNo'].str.startswith('C')].index, inplace=True)

In [ ]:
df[df['InvoiceNo'].str.startswith('C')].shape #Cancellation invoices were checked using the business rule (InvoiceNo starting with ‘C’). No such records were present in this dataset version, indicating cancellations were likely removed upstream.

In [ ]:
(df['UnitPrice']<=0).sum()

In [ ]:
df = df.drop(df[df['UnitPrice']<=0].index)

In [ ]:
(df['UnitPrice']<=0).sum()

In [ ]:
(df['Quantity']<=0).sum()

In [ ]:
df.drop(df[df['Quantity']<=0].index, inplace=True)

In [ ]:
(df['Quantity']<=0).sum() #Now it will wonn't affect on metrics

In [ ]:
df['Revenue'] = df['UnitPrice'] * df['Quantity']

In [ ]:
df.shape

In [ ]:
df.head()

### Fearure Engineering for cohort and retention

Goal:
*     Who are return users?
*     How often do customers return?
*     How does revenue evolve by cohort?

In [ ]:
df['OrderMonth'] = pd.to_datetime(df['InvoiceDate']).dt.to_period('M') # Truncate Date to month and year

In [ ]:
df['CohortMonth']= df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

In [ ]:
df.head(-20)

In [ ]:
df['CohortIndex']= (df['OrderMonth']-df['CohortMonth']).apply(lambda x: x.n) + 1

In [ ]:
df.head(-10)

In [ ]:
#df.loc('CohortIndex').apply(lambda x: x[x['CustomerID']==12680])[['CustomerID']].reset_index()
df.groupby('CohortIndex', group_keys=False).apply(lambda x: x[x['CustomerID']==12680])[['CustomerID','CohortIndex']].drop_duplicates().reset_index(drop=True)

### Retention Cohort Table
Goal:

*     How many user retain after first month?
*     How fast does retention decay?
*     Which cohorts are healthier?

In [ ]:
#Cohort Count table
cohort_counts= df.groupby(['CohortMonth','CohortIndex'])['CustomerID'].apply(pd.Series.nunique).reset_index()

In [ ]:
cohort_counts.rename(columns={'CustomerID':'CustomerCount'}, inplace=True)

In [ ]:
cohort_counts.shape

In [ ]:
cohort_counts.head(20)

In [ ]:
retention_table = cohort_counts.pivot_table(index='CohortMonth', columns='CohortIndex', values='CustomerCount')

In [ ]:
print(retention_table)

In [ ]:
#Converting Counts to Retention %
# retention_table = retention_table.applymap(lambda x: round(x*100,2))
cohort_sizes = retention_table.iloc[:, 0]
retention_matrix = retention_table.div(cohort_sizes, axis=0) * 100
retention_matrix = retention_matrix.round(2)
retention_matrix.columns = [f'Month_{i}' for i in retention_matrix.columns]
print(retention_matrix)

### Insights
Goal:
*     Clear insights

*     Actionable product recommendations

*     Business impact statements

1.
* Insight: Across almost all cohorts, retention drops sharply after Month 1 and falls below ~25% by Month 3, indicating that most customers do not form a repeat-purchase habit beyond their initial orders.
* Hypothesis: Customers complete one or two transactional purchases but lack strong re-engagement triggers such as reminders, replenishment needs, or personalized follow-ups, causing them to disengage early.
* Product Action:Introduce post-purchase lifecycle interventions such as:

    Automated email reminders within 14–30 days

    Personalized product recommendations based on first purchase

    Limited-time repeat-purchase incentives
* Expected Impact: Improving early lifecycle engagement could increase Month-3 retention by 5–10 percentage points, directly improving revenue per active customer.

2.
* Insight: Some early cohorts (e.g., 2010-12) show non-zero retention even after 10–12 months, suggesting the presence of a small but valuable group of long-lifecycle customers.
* Hypothesis: A subset of customers makes infrequent but recurring purchases driven by seasonal needs, gifting behavior, or specific product categories.
* Product Action: Identify customers with long gaps between purchases and:

    Segment them as “long-cycle customers”

    Trigger reactivation campaigns around expected reorder or seasonal windows
* Expected Impact: Reactivating even a small fraction of these customers can generate incremental revenue with low acquisition cost, improving long-term customer lifetime value (LTV).

3.
* Insight: Customers who remain active until Month 6 show significantly higher retention stability compared to early churners, indicating that retention beyond Month 3–4 strongly predicts long-term customer value.
* Hypothesis: Once customers cross a certain engagement threshold (multiple purchases across months), they develop trust, familiarity, or routine purchasing behavior, making them less likely to churn.
* Product Action: Design “early loyalty acceleration” strategies such as:

    Rewards after the 2nd or 3rd purchase

    Early-access offers for repeat buyers

    Explicit nudges encouraging a second and third order
* Expected Impact: Increasing the number of customers who reach Month-6 retention can disproportionately increase overall revenue, as these users contribute higher lifetime value compared to one-time buyers.

---------------------------------------------------------------**Thank You**------------------------------------------------------------------